In [ ]:
import time
import requests
import pandas as pd
from datetime import datetime, timedelta, timezone

# ---- config ----
API_KEY = "XXXX" # Replace with your actual API key
APP_KEY = "XXXX" # Replace with your actual application key
DAYS_BACK = 1
RESAMPLE = "15min"
BASE = "https://rt.ambientweather.net/v1"
AUTH = {"apiKey": API_KEY, "applicationKey": APP_KEY}

def get(url, params, retries=6):
    """GET with backoff — the API allows ~1 request/sec per API key."""
    for i in range(retries):
        r = requests.get(url, params=params, timeout=30)
        if r.status_code == 429:
            time.sleep(2 ** i)
            continue
        r.raise_for_status()
        return r.json()
    raise RuntimeError("Rate-limited too many times")

# ---- pick a device ----
devices = get(f"{BASE}/devices", AUTH)
for d in devices:
    print(d["macAddress"], "-", d.get("info", {}).get("name"))
mac = devices[0]["macAddress"]   # change if you have several stations

# ---- page backwards through history (max 288 records per call) ----
start_ms = (datetime.now(timezone.utc) - timedelta(days=DAYS_BACK)).timestamp() * 1000
end_ms, rows = None, []

while True:
    params = {**AUTH, "limit": 288}
    if end_ms is not None:
        params["endDate"] = int(end_ms)
    time.sleep(1.1)
    batch = get(f"{BASE}/devices/{mac}", params)
    if not batch:
        break
    rows.extend(batch)
    oldest = min(r["dateutc"] for r in batch)
    print(f"fetched {len(batch):>3} → back to {pd.to_datetime(oldest, unit='ms', utc=True)}")
    if oldest <= start_ms:
        break
    end_ms = oldest - 1

# ---- tidy up ----
df = (pd.DataFrame(rows)
        .drop_duplicates("dateutc")
        .assign(time=lambda x: pd.to_datetime(x["dateutc"], unit="ms", utc=True))
        .set_index("time")
        .sort_index())
df = df[df.index >= pd.to_datetime(start_ms, unit="ms", utc=True)]
raw = df.select_dtypes("number").drop(columns=["dateutc"], errors="ignore")

# ---- resample to 15 min with sensible aggregations ----
agg = {}
for c in raw.columns:
    if c in ("eventrainin", "dailyrainin", "weeklyrainin", "monthlyrainin",
             "yearlyrainin", "totalrainin"):
        agg[c] = "last"          # cumulative counters
    elif c in ("windgustmph", "maxdailygust"):
        agg[c] = "max"
    else:
        agg[c] = "mean"
df15 = raw.resample(RESAMPLE).agg(agg)
# ---- convert to Guatemala local time (UTC-6, no DST) ----
TZ = "America/Guatemala"
raw  = raw.tz_convert(TZ)
df15 = df15.tz_convert(TZ)
print(raw.shape, "raw ->", df15.shape, f"at {RESAMPLE}")
df15.tail()

In [2]:
df15.to_csv('data.csv')